In [ ]:
"""
Notebook: linha_de_base_horas.py
Projeto: Brasil Público — Escala 6x1
Etapa: 02_limpeza
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE        = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
ARQUIVO_IN  = BASE / "data/processed/pnadc_2023_anual.csv"
DIR_PROC    = BASE / "data/processed"
DIR_FIGS    = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df = pd.read_csv(ARQUIVO_IN, dtype=str)

for col in ["VD4031", "VD4035", "V2009", "V1028", "V4039"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

ocupados = df[
    (df["VD4002"].str.strip() == "1") &
    (df["VD4031"].notna()) &
    (df["VD4031"] > 0)
].copy()

print(f"Ocupados com horas válidas: {len(ocupados):,}")

setores = {
    "00": "Ativ. mal definidas",
    "10": "Agropecuária",
    "20": "Indústria geral",
    "30": "Construção",
    "40": "Comércio e rep.",
    "50": "Transp. e armaz.",
    "60": "Alojamento e alim.",
    "70": "Inf. e serv. prof.",
    "80": "Adm. públ. e educação",
    "81": "Saúde",
    "90": "Serv. domésticos",
}
ocupados["setor"] = ocupados["VD4010"].str.strip().map(setores)

bins   = [0, 20, 30, 39, 40, 44, 48, 60, 200]
labels = ["≤20h", "21–30h", "31–39h", "40h", "41–44h", "45–48h", "49–60h", "60h+"]
ocupados["faixa"] = pd.cut(ocupados["VD4031"], bins=bins, labels=labels, right=True)

ocupados["V4029"] = ocupados["V4029"].str.strip()
ocupados["vinculo"] = ocupados["V4029"].map({"1": "Formal (CLT)", "2": "Informal/sem carteira"})
ocupados["vinculo"] = ocupados["vinculo"].fillna("Outros/conta própria")

print("\nDistribuição por vínculo:")
print(ocupados["vinculo"].value_counts())

In [ ]:
metricas_setor = (
    ocupados.groupby("setor")["VD4031"]
    .agg(
        n="count",
        media="mean",
        mediana="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        pct_acima_44=lambda x: (x > 44).mean() * 100,
        pct_40_44=lambda x: ((x >= 40) & (x <= 44)).mean() * 100,
        pct_abaixo_40=lambda x: (x < 40).mean() * 100,
    )
    .round(2)
    .sort_values("media", ascending=False)
)

print(metricas_setor.to_string())
metricas_setor.to_csv(DIR_PROC / "horas_por_setor_2023.csv")
print("\n✓ Salvo: horas_por_setor_2023.csv")

In [ ]:
metricas_vinculo = (
    ocupados.groupby("vinculo")["VD4031"]
    .agg(
        n="count",
        media="mean",
        mediana="median",
        pct_acima_44=lambda x: (x > 44).mean() * 100,
        pct_40_44=lambda x: ((x >= 40) & (x <= 44)).mean() * 100,
        pct_abaixo_40=lambda x: (x < 40).mean() * 100,
    )
    .round(2)
)

print(metricas_vinculo.to_string())
metricas_vinculo.to_csv(DIR_PROC / "horas_por_vinculo_2023.csv")
print("\n✓ Salvo: horas_por_vinculo_2023.csv")

In [ ]:
faixas_setor = (
    ocupados.groupby(["setor", "faixa"])
    .size()
    .unstack(fill_value=0)
)
faixas_setor_pct = faixas_setor.div(faixas_setor.sum(axis=1), axis=0) * 100

print(faixas_setor_pct.round(1).to_string())
faixas_setor_pct.to_csv(DIR_PROC / "faixas_por_setor_2023.csv")
print("\n✓ Salvo: faixas_por_setor_2023.csv")

In [ ]:
setor_plot = metricas_setor.dropna().copy()
setor_plot = setor_plot[setor_plot.index != "Ativ. mal definidas"]
setor_plot = setor_plot.sort_values("media", ascending=True)

cores = [
    "#E84545" if v > 44 else "#F5A623" if v > 40 else "#2C6FBF"
    for v in setor_plot["media"]
]

fig, ax = plt.subplots(figsize=(11, 7))

bars = ax.barh(setor_plot.index, setor_plot["media"], color=cores, height=0.6)

ax.axvline(40, color="#2C6FBF", linewidth=1.5, linestyle="--", alpha=0.7, label="40h (proposta)")
ax.axvline(44, color="#F5A623", linewidth=1.5, linestyle="--", alpha=0.7, label="44h (CLT atual)")

for bar, (_, row) in zip(bars, setor_plot.iterrows()):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{row['media']:.1f}h  [P25:{row['p25']:.0f}–P75:{row['p75']:.0f}]",
        va="center", fontsize=8.5
    )

ax.set_xlabel("Média de horas habituais semanais", fontsize=11)
ax.set_title(
    "Jornada habitual por setor — Brasil 2023\n"
    "Média com P25–P75 · PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)
ax.set_xlim(0, 68)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(DIR_FIGS / "06_horas_setor_media_iqr.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 6 salvo.")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

ordem = faixas_setor_pct.drop("Ativ. mal definidas", errors="ignore")
ordem = ordem.loc[ordem.sum(axis=1) > 0]
ordem = ordem.sort_values("41–44h", ascending=False)

data = ordem.values.astype(float)
im   = ax.imshow(data, cmap="YlOrRd", aspect="auto", vmin=0, vmax=40)

ax.set_xticks(range(len(ordem.columns)))
ax.set_xticklabels(ordem.columns, rotation=35, ha="right", fontsize=10)
ax.set_yticks(range(len(ordem.index)))
ax.set_yticklabels(ordem.index, fontsize=10)

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        cor = "white" if val > 25 else "black"
        ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=8, color=cor)

plt.colorbar(im, ax=ax, label="% dos trabalhadores do setor")
ax.set_title(
    "Distribuição de faixas de jornada por setor — Brasil 2023\n"
    "% dos ocupados em cada setor · PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "07_heatmap_faixas_setor.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 7 salvo.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

vinculos = ["Formal (CLT)", "Informal/sem carteira", "Outros/conta própria"]
cores_v  = ["#2C6FBF", "#E84545", "#F5A623"]

for vinculo, cor in zip(vinculos, cores_v):
    sub = ocupados[ocupados["vinculo"] == vinculo]["VD4031"].dropna()
    if len(sub) > 100:
        ax.hist(
            sub,
            bins=range(1, 80, 2),
            density=True,
            alpha=0.55,
            color=cor,
            label=f"{vinculo} (n={len(sub):,})"
        )

ax.axvline(40, color="black", linewidth=1.5, linestyle="--", label="40h")
ax.axvline(44, color="gray",  linewidth=1.5, linestyle=":",  label="44h")

ax.set_xlabel("Horas habituais semanais", fontsize=11)
ax.set_ylabel("Densidade", fontsize=11)
ax.set_title(
    "Distribuição de jornada: formal vs informal — Brasil 2023\n"
    "PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)
ax.set_xlim(0, 80)
ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig(DIR_FIGS / "08_formal_informal_jornada.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 8 salvo.")

In [ ]:
print("\n══ RESUMO DA LINHA DE BASE ══════════════════════════════")
print(f"Total ocupados analisados:        {len(ocupados):,}")
print(f"Setores com dados:                {ocupados['setor'].notna().sum():,} obs em {ocupados['setor'].nunique()} setores")
print(f"\nJornada média geral:              {ocupados['VD4031'].mean():.1f}h/semana")
print(f"Mediana geral:                    {ocupados['VD4031'].median():.0f}h/semana")
print(f"\n% trabalhando ≤ 40h:              {(ocupados['VD4031'] <= 40).mean()*100:.1f}%")
print(f"% trabalhando 41–44h:             {((ocupados['VD4031'] > 40) & (ocupados['VD4031'] <= 44)).mean()*100:.1f}%")
print(f"% trabalhando > 44h:              {(ocupados['VD4031'] > 44).mean()*100:.1f}%")
print(f"\nSetor com maior jornada média:    {metricas_setor['media'].idxmax()} ({metricas_setor['media'].max():.1f}h)")
print(f"Setor com menor jornada média:    {metricas_setor['media'].dropna().idxmin()} ({metricas_setor['media'].dropna().min():.1f}h)")
print(f"\nArquivos salvos em data/processed/ e outputs/figures/")